# 8 — Estrazione embedding testuali del vocabolario (concept naming)

Notebook per la fase di **concept naming** del MSAE.

**Cosa fa**: estrae gli embedding CLIP (ViT-L/14, 768D) per i ~37k termini del vocabolario LAION-400M unigram presenti nella repo MSAE (`vocab/laion_400_unigram.txt`).

**Perché**: nello step successivo  ogni feature del decoder MSAE viene "nominata" trovando il termine del vocabolario con maggior cosine similarity rispetto alla direzione della feature nello spazio CLIP.



## 1. Setup — Drive e clone della repo MSAE

Monta Google Drive e clona la repo ufficiale [WolodjaZ/MSAE](https://github.com/WolodjaZ/MSAE). La repo contiene il file vocabolario `vocab/laion_400_unigram.txt` e le utility `EmbeddingExtractor` + `load_vocab` che riutilizzeremo direttamente.

In [ ]:
from google.colab import drive
import pathlib

# Mount Drive con gestione errore esplicita
try:
    drive.mount('/content/drive')
except Exception as e:
    raise RuntimeError(f"Mount Drive fallito: {e}")

DRIVE = pathlib.Path("/content/drive/MyDrive/tesi")
if not DRIVE.exists():
    raise FileNotFoundError(
        f"Cartella Drive non trovata: {DRIVE}\n"
        f"Verifica che la cartella MyDrive/tesi/ esista sul tuo Drive."
    )
print(f"Drive OK: {DRIVE}")

# Clone della repo MSAE 
!git clone https://github.com/WolodjaZ/MSAE.git /content/MSAE 2>/dev/null || echo "Repo MSAE già presente"

%cd /content/MSAE

# Verifica presenza del vocab file
VOCAB_FILE = pathlib.Path("/content/MSAE/vocab/laion_400_unigram.txt")
if not VOCAB_FILE.exists():
    raise FileNotFoundError(
        f"Vocab file non trovato: {VOCAB_FILE}\n"
        f"Il clone della repo MSAE potrebbe essere fallito."
    )
print(f"Vocab file OK: {VOCAB_FILE}")

## 2. Installazione di open_clip e caricamento del modello

Installa la libreria `open_clip_torch` (per il modello CommonPool XL) e il pacchetto OpenAI `clip` (necessario solo per poter importare `precompute_activations.py` della repo MSAE, dato che quel modulo fa `import clip` a livello top-level).

Carica **CLIP-ViT-L-14-CommonPool.XL** (pre-addestrato su CommonPool XL, 13B campioni) sulla GPU T4 tramite `open_clip`. Da `precompute_activations` riusiamo solo `load_vocab` per leggere il vocabolario (supporta `laion_unigram`, `laion_bigram`, ecc.).

In [ ]:
!pip install open_clip_torch -q
# clip OpenAI serve solo per poter importare precompute_activations (load_vocab),
# NON viene usato per il modello: il modello è caricato via open_clip
!pip install git+https://github.com/openai/CLIP.git -q

import sys
sys.path.insert(0, '/content/MSAE')

# Dalla repo MSAE usiamo solo load_vocab per leggere il vocabolario
# (supporta laion_unigram, laion_bigram, ecc.)
from precompute_activations import load_vocab

import open_clip
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type != "cuda":
    print("ATTENZIONE: GPU non rilevata. Cambia runtime a T4 (Runtime -> Change runtime type -> GPU).")

# Caricamento del modello CommonPool XL tramite open_clip
model, _, _ = open_clip.create_model_and_transforms(
    'hf-hub:laion/CLIP-ViT-L-14-CommonPool.XL.clip-s13B-b90K'
)
model = model.to(device).eval()
tokenizer = open_clip.get_tokenizer('hf-hub:laion/CLIP-ViT-L-14-CommonPool.XL.clip-s13B-b90K')

print("Modello caricato: CLIP-ViT-L-14-CommonPool.XL (embedding dim = 768)")

## 3. Caricamento del vocabolario

Riusiamo `load_vocab("laion_unigram", vocab_size=-1)` da `precompute_activations.py`: legge `vocab/laion_400_unigram.txt` riga per riga e fa `strip()`. Aggiungiamo un secondo filtro per scartare eventuali righe vuote (anche se il file è già pulito).



In [ ]:
# -1 = carica tutto il vocabolario
vocab_dataset = load_vocab("laion_unigram", vocab_size=-1)
terms_raw = vocab_dataset.data

# Strip + filtro righe vuote 
terms = [t.strip() for t in terms_raw if t.strip()]

N = len(terms)
print(f"Vocabolario caricato: {N} termini")
print(f"Primi 10 termini: {terms[:10]}")
print(f"Ultimi 5 termini: {terms[-5:]}")

## 4. Generazione degli embedding CLIP

Per ogni termine del vocabolario:
1. **Tokenizzazione**: `clip.tokenize(term, truncate=True)` → tensore di indici (max 77 token, gestito da `embed_text`)
2. **Encoding**: `model.encode_text(tokens)` in autocast fp16 → vettore 768D
3. **L2-normalizzazione**: norma unitaria per ogni vettore 
Processiamo in batch da **512 termini** 

In [ ]:
BATCH_SIZE = 512
D = 768  # ViT-L/14 embedding dim

embeddings = np.zeros((N, D), dtype=np.float32)

print("Generazione embedding in corso...")
for start in tqdm(range(0, N, BATCH_SIZE), desc="Batch"):
    end = min(start + BATCH_SIZE, N)
    batch_terms = terms[start:end]

    # Tokenizzazione con open_clip (truncate automatico a 77 token)
    tokens = tokenizer(batch_terms).to(device)

    with torch.no_grad():
        text_features = model.encode_text(tokens)

    # fp32 + L2 normalize
    text_features = text_features.float()
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    embeddings[start:end] = text_features.cpu().numpy()

print(f"\nEstrazione completata. Shape finale: {embeddings.shape}")

## 5. Salvataggio su Drive e sanity check

Salva due file allineati su Drive (`MyDrive/tesi/`):

- **`laion_unigram_ViT-L~14_text_{N}_{D}.npy`** — embedding in formato **raw memmap** (no header numpy). Il naming segue la convenzione `SAEDataset` della repo MSAE (`..._{N}_{D}.npy`), così il prossimo step (`sae_naming.py`) lo carica direttamente parsando dimensioni dal nome file.
- **`laion_unigram_ViT-L~14_text_{N}.txt`** — termini in ordine **identico** agli embedding (indice riga = indice embedding), per recuperare il nome del concetto associato a ciascuna feature SAE.

Sanity check: shape, dtype, norma media (deve essere ~1.0 dopo L2-norm), primi 5 termini con i loro embedding.

In [ ]:
OUT_DIR = DRIVE
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Naming compatibile con SAEDataset: parsing fa split('_')[-2:] -> (N, D)
EMB_PATH   = OUT_DIR / f"laion_unigram_ViT-L~14_text_{N}_{D}.npy"
TERMS_PATH = OUT_DIR / f"laion_unigram_ViT-L~14_text_{N}.txt"

# Salvataggio embedding come raw memmap (no header numpy) — formato richiesto da SAEDataset
mm = np.memmap(EMB_PATH, dtype=np.float32, mode='w+', shape=embeddings.shape)
mm[:] = embeddings
mm.flush()
del mm

# Salvataggio termini allineati (un termine per riga, stesso ordine degli embedding)
with open(TERMS_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(terms))

# --- Sanity check finale ---
print(f"Shape embedding: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")
norm_mean = np.linalg.norm(embeddings, axis=1).mean()
ok = "OK" if abs(norm_mean - 1.0) < 1e-3 else "FAIL"
print(f"Norma media vettori: {norm_mean:.4f}  [{ok}]")
print(f"Primi 5 termini: {terms[:5]}")
print()
print("Primi 3 valori dei primi 5 embedding:")
for i in range(5):
    print(f"  {terms[i]:>20s} -> {embeddings[i, :3]}")
print()
print("File salvati:")
print(f"  - {EMB_PATH}  ({EMB_PATH.stat().st_size / 1e6:.1f} MB)")
print(f"  - {TERMS_PATH}  ({TERMS_PATH.stat().st_size / 1e3:.1f} KB)")